In [ ]:
!git clone -q --depth 1 --filter=blob:none --sparse -b claude/audio-round4-2026-11-17 https://github.com/ALENZACODES/Al-Heb.git
!cd Al-Heb && git sparse-checkout set tools data/corpus/torah audio/m/israeli/line audio/f/israeli/line src && git log -1 --format='%h %s'


In [ ]:
!pip install -q faster-whisper==1.2.1 ctranslate2 nvidia-cublas-cu12 'nvidia-cudnn-cu12==9.*' 2>&1 | tail -1
!ct2-transformers-converter --model renikud/whisper-he-ipa --revision e709a907f8e0e6ebcb83340b8563818dd1bc3bee --output_dir Al-Heb/phoneme-ct2 --quantization int8 --copy_files tokenizer.json preprocessor_config.json 2>&1 | tail -1


In [ ]:
import os, glob, subprocess
lib = ':'.join(glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/*/lib') + glob.glob('/usr/lib/python3*/site-packages/nvidia/*/lib'))
env = dict(os.environ, ALEF_DEVICE='cuda', LD_LIBRARY_PATH=lib + ':' + os.environ.get('LD_LIBRARY_PATH', ''))
os.makedirs('/kaggle/working/check', exist_ok=True)
procs = []
for gpu, shards in ((0, (20, 21)), (1, (22, 23))):
    cmd = ' && '.join(f'python tools/torah/audio/check_shard.py --shard {s} --of 24 --model phoneme-ct2 --out /kaggle/working/check/shard-{s}.jsonl' for s in shards)
    procs.append(subprocess.Popen(cmd, shell=True, cwd='Al-Heb', env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu))))
print([p.wait() for p in procs])


In [ ]:
!cd Al-Heb && python tools/torah/audio/check_summary.py /kaggle/working/check/shard-*.jsonl --fails /kaggle/working/fails.txt --md /kaggle/working/TORAH-CHECK.md && cat /kaggle/working/TORAH-CHECK.md
